In [ ]:
"""
PIPELINE BASELINE - Tratamento BPM sem otimizacoes manuais.

Fluxo: XLSX -> CSV -> join comum -> filtro tardio -> agregacao -> Parquet.
Execute na EC2 com: python3 -i tratamento_batimentos_baseline.py
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, max, min, round as spark_round
from pyspark.sql.functions import stddev, sum as spark_sum, when
from pyspark.sql.types import BooleanType, DoubleType, IntegerType, StringType
from pyspark.sql.types import StructField, StructType
import os
import time


spark = (SparkSession.builder
    .appName("HajaCoracao-BPM-Baseline")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "6")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast automatico:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Spark UI: http://localhost:4040")

inicio = time.time()
xlsx = "/home/ubuntu/dados_batimentos.xlsx"
csv = "/home/ubuntu/dados_batimentos_baseline.csv"

if not os.path.exists(csv):
    import pandas as pd
    pd.read_excel(xlsx, sheet_name="Dados Batimentos").to_csv(csv, index=False)

schema = StructType([
    StructField("messageId", IntegerType(), False),
    StructField("deviceId", StringType(), False),
    StructField("heartRate", DoubleType(), False),
    StructField("heartRateTarget", DoubleType(), False),
    StructField("activityState", IntegerType(), False),
    StructField("activityLabel", StringType(), False),
    StructField("bpmAlert", BooleanType(), False),
    StructField("timestamp", StringType(), False),
    StructField("deviceIndex", IntegerType(), False),
    StructField("timeSinceStart", StringType(), False),
])

# A baseline mantém todas as colunas e lê todo o conjunto antes de reduzir dados.
bpm = spark.read.option("header", True).schema(schema).csv(csv)
dim_devices = spark.createDataFrame([
    ("device-01",), ("device-02",), ("device-03",),
    ("device-04",), ("device-05",), ("device-06",),
], ["deviceId"])

# Método ineficiente de propósito:
# 1. O join comum pode redistribuir os dois lados por deviceId (Exchange).
# 2. O filtro acontece depois do join, então dados inválidos já participaram
#    da etapa de shuffle.
# 3. A dimensão pequena não usa broadcast e também pode gerar shuffle.
bpm_completo = bpm.join(dim_devices, on="deviceId", how="inner")
bpm_filtrado = (bpm_completo
    .filter(col("heartRate").isNotNull())
    .filter((col("heartRate") >= 30) & (col("heartRate") <= 220))
    .withColumn("timestamp", col("timestamp").cast("timestamp")))

resultado = (bpm_filtrado
    # O groupBy é uma operação wide: registros da mesma chave precisam ser
    # reunidos, gerando o Exchange necessário para a agregação.
    .groupBy("deviceId", "activityLabel")
    .agg(
        count("*").alias("total_registros"),
        spark_round(avg("heartRate"), 2).alias("media_bpm"),
        spark_round(stddev("heartRate"), 2).alias("desvio_bpm"),
        min("heartRate").alias("min_bpm"),
        max("heartRate").alias("max_bpm"),
        spark_sum(when(col("bpmAlert") == True, 1).otherwise(0)).alias("alertas"),
    ))

print("=== PLANO BASELINE ===")
# O explain permite localizar Exchange e comparar o custo do plano físico.
resultado.explain("formatted")
print("=== RESULTADO ===")
# O orderBy gera outro Exchange, mantido aqui para a comparação ser igual
# à versão eficiente.
resultado.orderBy("deviceId", "activityLabel").show(20, truncate=False)

resultado.write.mode("overwrite").parquet("/home/ubuntu/processed_bpm_baseline_parquet")
print(f"Tempo total de execucao: {time.time() - inicio:.2f} segundos")
print("Spark UI ativa em http://localhost:4040")